In [1]:
import os
import pandas as pd
import re

In [2]:
def extract_all_molecular_info(log_folder_path):   
    # log_folder_path: 存放 .log 文件的文件夹路径
    filename_list = os.listdir(log_folder_path)
    all_data = []

    # 定义提取能量的正则 (匹配 SCF Done 行)
    energy_pattern = re.compile(r'SCF Done:\s+E\([^\)]+\)\s+=\s+(-?\d+\.\d+(?:[DE][-+]\d+)?)')
    for name in filename_list:
        file_path = os.path.join(log_folder_path, name)
        # 仅处理 .log 文件
        if os.path.isfile(file_path) and name.endswith('.log'):
            with open(file_path, 'r') as file:
                lines = file.readlines()
                # 将所有行合并成一个字符串，方便正则提取能量
                full_content = ''.join(lines)

                # -----------------------------
                # 1. 提取 Energy (最后一次 SCF Done)
                # -----------------------------
                energy_matches = energy_pattern.findall(full_content)
                if energy_matches:
                    # 取最后一个匹配值，并将 Gaussian 的 'D' 替换为 Python 能识别的 'E'
                    scf_energy = float(energy_matches[-1].replace('D', 'E'))
                else:
                    scf_energy = None

                # -----------------------------
                # 2. 提取 HOMO, LUMO, Gap
                # -----------------------------
                homo = None
                lumo = None
                gap = None

                # 倒序查找最后一个包含 'Alpha  occ. eigenvalues' 的行
                last_occ_index = None
                for i in range(len(lines)-1, -1, -1):
                    if 'Alpha  occ. eigenvalues' in lines[i]:
                        last_occ_index = i
                        break # 找到后立即退出循环
                
                if last_occ_index is not None:
                    # --- 获取 HOMO ---
                    occ_line = lines[last_occ_index].strip()
                    # Gaussian输出通常为: Alpha  occ. eigenvalues --  -0.3  -0.2
                    # split()后，第5个元素(索引4)开始是数值
                    occ_values = occ_line.split()[4:] 
                    if occ_values:
                        homo = float(occ_values[-1]) # 最后一个占据轨道能量

                    # --- 获取 LUMO ---
                    # 检查下一行是否是 'Alpha virt. eigenvalues'
                    if last_occ_index + 1 < len(lines):
                        virt_line = lines[last_occ_index + 1].strip()
                        if 'Alpha virt. eigenvalues' in virt_line:
                            virt_values = virt_line.split()[4:]
                            if virt_values:
                                lumo = float(virt_values[0]) # 第一个空轨道能量

                    # --- 计算 Gap ---
                    if homo is not None and lumo is not None:
                        gap = lumo - homo

                # -----------------------------
                # 3. 整合数据
                # -----------------------------
                filename_no_ext = name.rsplit('.', 1)[0]
                
                all_data.append({
                    'Filename': filename_no_ext,
                    'Energy': scf_energy,
                    'HOMO': homo,
                    'LUMO': lumo,
                    'Gap': gap
                })
    
    # 生成 DataFrame 并保存
    df = pd.DataFrame(all_data)
    print(f"处理完成，共提取 {len(df)} 个文件，结果已保存为 global_info.csv")
    return df

In [7]:
df_cat = extract_all_molecular_info('./data/round_2/predict/6new_pre_data/logs/cat/')
df_cat.to_csv('./data/round_2/predict/6new_pre_data/logs/global_info_cat.csv', index=False)

处理完成，共提取 20 个文件，结果已保存为 global_info.csv


In [8]:
df_lig = extract_all_molecular_info('./data/round_2/predict/6new_pre_data/logs/ligand/')
df_lig.to_csv('./data/round_2/predict/6new_pre_data/logs/global_info_lig.csv', index=False)

处理完成，共提取 218 个文件，结果已保存为 global_info.csv


In [9]:
df_cat.shape

(20, 5)

In [10]:
df_lig.shape

(218, 5)